# HDB Resale Price Regression — Notebook 7: Period Comparison

We run the same Model 10 on two consecutive 2-year windows and compare coefficients:

- **Period A:** May 2022 – April 2024 (52,773 transactions)
- **Period B:** May 2024 – April 2026 (50,718 transactions) — our main analysis

**What this tests:**
- Are the model's findings stable across time, or specific to the current period?
- Have superstition effects grown? (ties to the finding that 8-ending prices surged from near-zero to 10%)
- Have geographic premiums shifted? (new MRT lines, changing estate maturity)
- Is the model itself robust, or is it overfitting to one period's quirks?

In [1]:
import pandas as pd
import numpy as np
import os, requests, time, re

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

print('Setup complete.')

Setup complete.


## Build prior-period dataset (May 2022 – April 2024)

We need to:
1. Filter the full 1990–2026 data to the prior 2-year window
2. Geocode any blocks not already in our cache
3. Compute the same supplementary variables (distances, superstition, timing)

In [2]:
# Load full dataset and filter to prior period
full = pd.read_csv('data/hdb_resale_full.csv', low_memory=False)
full['month'] = pd.to_datetime(full['month'])

prior = full[(full['month'] >= '2022-05-01') & (full['month'] <= '2024-04-01')].copy()
print(f'Prior period: {prior["month"].min().strftime("%Y-%m")} to {prior["month"].max().strftime("%Y-%m")}')
print(f'Rows: {len(prior):,}')

Prior period: 2022-05 to 2024-04
Rows: 52,773


In [3]:
# Geocode any blocks not already cached
geo = pd.read_csv('data/onemap_geocoded.csv')
prior_blocks = prior[['block', 'street_name']].drop_duplicates()

existing = set(zip(geo['block'], geo['street_name']))
new_blocks = prior_blocks[~prior_blocks.apply(lambda r: (r['block'], r['street_name']) in existing, axis=1)]
print(f'Blocks to geocode: {len(new_blocks):,}')

if len(new_blocks) > 0:
    results = []
    for _, row in new_blocks.iterrows():
        query = f"{row['block']} {row['street_name']}"
        try:
            resp = requests.get(
                'https://www.onemap.gov.sg/api/common/elastic/search',
                params={'searchVal': query, 'returnGeom': 'Y', 'getAddrDetails': 'Y'},
                timeout=10
            )
            data = resp.json()
            if data.get('found', 0) > 0:
                r = data['results'][0]
                results.append({'block': row['block'], 'street_name': row['street_name'],
                    'latitude': float(r['LATITUDE']), 'longitude': float(r['LONGITUDE'])})
            else:
                results.append({'block': row['block'], 'street_name': row['street_name'],
                    'latitude': None, 'longitude': None})
        except:
            results.append({'block': row['block'], 'street_name': row['street_name'],
                'latitude': None, 'longitude': None})
        time.sleep(0.3)
    
    new_geo = pd.DataFrame(results)
    geo = pd.concat([geo, new_geo], ignore_index=True)
    geo.to_csv('data/onemap_geocoded.csv', index=False)
    print(f'Geocoded {new_geo["latitude"].notna().sum():,} new blocks, total now {len(geo):,}')
else:
    print('All blocks already geocoded.')

Blocks to geocode: 0
All blocks already geocoded.


In [4]:
# Compute supplementary variables for prior period
# (same logic as Notebook 3)

# Join geocoding
prior = prior.merge(geo[['block', 'street_name', 'latitude', 'longitude']],
                    on=['block', 'street_name'], how='left')
print(f'Geocoded: {prior["latitude"].notna().sum():,} / {len(prior):,} ({prior["latitude"].notna().mean()*100:.1f}%)')

# Load location reference files
loc_files = {
    'mrt': 'data/mrt_stations.csv', 'school': 'data/schools.csv',
    'hawker': 'data/loc_hawker_centres.csv', 'park': 'data/loc_parks.csv',
    'hospital': 'data/loc_hospitals.csv', 'columbarium': 'data/loc_columbarium.csv',
    'temple': 'data/loc_temples.csv', 'coast': 'data/loc_coast.csv',
    'popular_school': 'data/loc_popular_schools.csv',
}

# Compute distances per block
block_coords = prior[['block', 'street_name', 'latitude', 'longitude']].drop_duplicates().dropna(subset=['latitude'])
print(f'Computing distances for {len(block_coords):,} blocks...')

CBD_LAT, CBD_LON = 1.2840, 103.8514
block_coords['dist_cbd_km'] = haversine_km(block_coords['latitude'], block_coords['longitude'], CBD_LAT, CBD_LON)

for key, path in loc_files.items():
    ref = pd.read_csv(path)
    ref_lats, ref_lons, ref_names = ref['latitude'].values, ref['longitude'].values, ref['name'].values
    dists, names = [], []
    for _, row in block_coords.iterrows():
        d = haversine_km(row['latitude'], row['longitude'], ref_lats, ref_lons) * 1000
        idx = np.argmin(d)
        dists.append(d[idx])
        names.append(ref_names[idx])
    block_coords[f'{key}_dist_m'] = dists

# Join distances back
dist_cols = [c for c in block_coords.columns if '_dist_m' in c or 'dist_cbd' in c]
prior = prior.merge(block_coords[['block', 'street_name'] + dist_cols],
                    on=['block', 'street_name'], how='left')

# Superstition variables
price_str = prior['resale_price'].astype(int).astype(str)
prior['num_eights_tail'] = price_str.str[-4:].str.count('8')
prior['price_has_168'] = price_str.str.contains('168').astype(int)
prior['block_has_4'] = prior['block'].str.contains('4').astype(int)

# CNY month
cny_ranges = [('2022-02-01', '2022-03-02'), ('2023-01-22', '2023-02-19'), ('2024-02-10', '2024-03-09')]
prior['cny_month'] = 0
for start, end in cny_ranges:
    mask = (prior['month'] >= start) & (prior['month'] <= end)
    prior.loc[mask, 'cny_month'] = 1

# Flat model grouping
model_counts = prior['flat_model'].value_counts()
rare = model_counts[model_counts < 50].index.tolist()
prior['flat_model_grouped'] = prior['flat_model'].apply(lambda x: 'Other' if x in rare else x)

# Quadratic lease + month factor
prior['remaining_lease_sq'] = prior['remaining_lease_years'] ** 2
prior['month_factor'] = prior['month'].dt.strftime('%Y-%m')

print(f'\nPrior period dataset ready: {len(prior):,} rows, {len(prior.columns)} columns')
prior.to_csv('data/hdb_prior_period.csv', index=False)
print('Saved to data/hdb_prior_period.csv')

Geocoded: 51,548 / 52,773 (97.7%)
Computing distances for 8,872 blocks...

Prior period dataset ready: 52,773 rows, 36 columns
Saved to data/hdb_prior_period.csv


## Run Model 10 on both periods

In [5]:
%load_ext rpy2.ipython
import warnings
warnings.filterwarnings('ignore')

Error importing in API mode: ImportError("dlopen(/Users/wongpeiting/.pyenv/versions/3.13.9/lib/python3.13/site-packages/_rinterface_cffi_api.abi3.so, 0x0002): Library not loaded: /Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib\n  Referenced from: <B96A8100-FA7A-3EFC-8726-931D26646DE6> /Users/wongpeiting/.pyenv/versions/3.13.9/lib/python3.13/site-packages/_rinterface_cffi_api.abi3.so\n  Reason: tried: '/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file), '/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file)")
Trying to import in ABI mode.


In [6]:
%%R
library(tidyverse)
library(scales)
library(sandwich)
library(lmtest)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.6
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.1     ✔ tibble    3.3.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors



Attaching package: ‘scales’

The following object is masked from ‘package:purrr’:

    discard

The following object is masked from ‘package:readr’:

    col_factor

Loading required package: zoo

Attaching package: ‘zoo’

The following objects are masked from ‘package:base’:

    as.Date, as.Date.numeric



In [7]:
%%R
# Load both periods
current <- read_csv('data/hdb_analysis.csv', show_col_types = FALSE)
prior <- read_csv('data/hdb_prior_period.csv', show_col_types = FALSE)

current$remaining_lease_sq <- current$remaining_lease_years^2
current$month_factor <- factor(format(current$month, '%Y-%m'))
prior$month_factor <- factor(prior$month_factor)

cat(sprintf('Current period: %s rows\n', comma(nrow(current))))
cat(sprintf('Prior period:   %s rows\n', comma(nrow(prior))))

Current period: 50,718 rows
Prior period:   52,773 rows


In [8]:
%%R
# Same Model 10 formula for both periods
model_formula <- resale_price ~ town + flat_type + floor_area_sqm + storey_mid +
    remaining_lease_years + remaining_lease_sq +
    flat_model_grouped +
    dist_cbd_km + mrt_dist_m + hawker_dist_m +
    popular_school_dist_m +
    park_dist_m + hospital_dist_m +
    columbarium_dist_m + temple_dist_m +
    coast_dist_m +
    num_eights_tail +
    price_has_168 +
    block_has_4 +
    cny_month +
    month_factor

m_current <- lm(model_formula, data = current)
m_prior <- lm(model_formula, data = prior)

cat(sprintf('Current period R-squared: %.4f\n', summary(m_current)$r.squared))
cat(sprintf('Prior period R-squared:   %.4f\n', summary(m_prior)$r.squared))

Current period R-squared: 0.9018
Prior period R-squared:   0.9057


## Side-by-side coefficient comparison

In [9]:
%%R
# Compare key coefficients between periods
key_vars <- c('floor_area_sqm', 'storey_mid', 'remaining_lease_years', 'remaining_lease_sq',
              'dist_cbd_km', 'mrt_dist_m', 'hawker_dist_m', 'popular_school_dist_m',
              'park_dist_m', 'hospital_dist_m',
              'columbarium_dist_m', 'temple_dist_m', 'coast_dist_m',
              'num_eights_tail', 'price_has_168', 'block_has_4', 'cny_month')

robust_current <- coeftest(m_current, vcov = vcovHC(m_current, type = 'HC1'))
robust_prior <- coeftest(m_prior, vcov = vcovHC(m_prior, type = 'HC1'))

cat(sprintf('%-25s %15s %15s %15s\n', 'Variable', '2022-2024', '2024-2026', 'Change'))
cat(paste(rep('-', 73), collapse = ''), '\n')

for (v in key_vars) {
  if (v %in% rownames(robust_prior) & v %in% rownames(robust_current)) {
    c_prior <- coef(m_prior)[v]
    c_current <- coef(m_current)[v]
    change <- c_current - c_prior
    
    p_prior <- robust_prior[v, 'Pr(>|t|)']
    p_current <- robust_current[v, 'Pr(>|t|)']
    
    sig_prior <- ifelse(p_prior < 0.05, '*', '')
    sig_current <- ifelse(p_current < 0.05, '*', '')
    
    cat(sprintf('%-25s %14s%s %14s%s %+14s\n', v,
        comma(round(c_prior)), sig_prior,
        comma(round(c_current)), sig_current,
        comma(round(change))))
  }
}

Variable                        2022-2024       2024-2026          Change
------------------------------------------------------------------------- 
floor_area_sqm                     5,039*          5,568*            529
storey_mid                         5,006*          5,399*            393
remaining_lease_years              6,141*         11,072*          4,931
remaining_lease_sq                    -2            -29*            -27
dist_cbd_km                      -10,746*        -16,114*         -5,368
mrt_dist_m                           -63*            -79*            -16
hawker_dist_m                        -15*            -20*             -5
popular_school_dist_m                -13*            -10*              3
park_dist_m                            7*              3*             -4
hospital_dist_m                        3*              4*              1
columbarium_dist_m                     6*              8*              3
temple_dist_m                        -20*        

## Interpretation

The model is **stable across time**: R² = 0.906 in the prior period vs 0.902 in the current period. The same variables explain the same share of price variation — these findings are not artifacts of one period's quirks. But the *size* of several coefficients shifted meaningfully.

### Lease decay awareness has surged

The biggest shift in the table. The linear lease term nearly doubled ($6,141 → $11,072) and the quadratic term went from insignificant to significant (-$2 → -$29). Buyers are pricing remaining lease much more aggressively than two years ago — consistent with sustained policy messaging since PM Lee's 2017 NDR speech on lease decay.

### The centrality premium is growing

Distance from CBD: -$10,746 → -$16,114 per km (up 50%). Distance from MRT: -$63 → -$79 per metre (up 25%). The premium for central, well-connected locations has widened considerably — possibly reflecting the post-COVID return-to-office trend, or simply that central-area supply is fixed while demand grows.

Hawker centre proximity also strengthened (-$15 → -$20 per metre), while popular school proximity held steady (-$13 → -$10).

### Superstition effects are shifting in form

- **Block 4 penalty nearly doubled**: -$5,952 → -$10,133. Buyers are increasingly willing to pay more to avoid blocks with the digit 4.
- **"168" premium surged from nothing to $32,696**: In 2022–2024, pricing a flat at a number containing "168" (一路发, "prosperity all the way") had no statistically significant effect. Two years later, it commands a $33K premium — sellers have learned the trick works, and buyers are responding.
- **Per-digit-8 premium fell**: $1,742 → $1,124. The per-digit effect of 8s shrank, but the more deliberate "168" signal grew massively. The market may be shifting from diffuse 8-preference to targeted auspicious-number strategies.

Overall, superstition effects are growing in magnitude even as they evolve in form.

### CNY premium shrank

$73,772 → $59,316. Still large and significant, but smaller in the current period — could reflect timing differences between windows or regression toward the mean after an unusually strong CNY effect in the prior period.

### Physical attributes are stable

Floor area ($5,039 → $5,568 per sqm) and storey ($5,006 → $5,399 per floor) both increased modestly, roughly in line with general price inflation. These fundamentals don't change much between periods.

### Summary

What changed is *how much* the market rewards centrality and remaining lease, and *how much* it penalises superstitious red flags. The coefficients that shifted most — lease decay, CBD distance, block-4 penalty, 168 premium — all moved in directions consistent with a market that is simultaneously more sophisticated about fundamentals and more susceptible to behavioural pricing.